In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from gpt_model3 import GPTModel3
from gpt_model1 import TokenDataset
import requests
from torchinfo import summary

# gpt 2 toenizer
from transformers import GPT2Tokenizer

In [2]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

In [3]:
# tokenize the text
text = requests.get('https://www.gutenberg.org/files/35/35-0.txt').text

# text needs to be pytorch tensors
tokens = tokenizer.encode(text)
print(f'Variable "tokens" is type {type(tokens)}')

# convert to pytorch
tmTokens = torch.tensor( tokens )
print(f'Variable "tmTokens" is type {type(tmTokens)} and has {len(tmTokens)}')
print(tmTokens.shape)

Token indices sequence length is longer than the specified maximum sequence length for this model (48533 > 1024). Running this sequence through the model will result in indexing errors


Variable "tokens" is type <class 'list'>
Variable "tmTokens" is type <class 'torch.Tensor'> and has 48533
torch.Size([48533])


In [4]:
# Hyper parameters
seq_len = 8 # aka context length
stride = 2
n_vocab = tokenizer.vocab_size
print("Vocab size = ", n_vocab)

# model hyperparameters
embed_dim = 2**6 # 64

batch_size = 5

Vocab size =  50257


In [5]:
##### TRANSPOSE EAMPLE #####
k = torch.randn((batch_size, seq_len, embed_dim))
print(k.shape)
print(k.mT.shape)
print(k.transpose(1,2).shape)
print(k.transpose(-2,-1).shape) # Tells to transpose last two dimensions -2 => seq_len, -1 => embed_dim

torch.Size([5, 8, 64])
torch.Size([5, 64, 8])
torch.Size([5, 64, 8])
torch.Size([5, 64, 8])


In [1]:
token_dataset = TokenDataset(tokenizer, text, seq_len, stride)
# print(len(token_dataset))
token_dataset[4]

dataloader = DataLoader(
                token_dataset,
                batch_size = batch_size,
                shuffle    = True,
                drop_last  = True
            )

# let's have a look at the indices
X,y = next(iter(dataloader))
print("X shape ", X.shape)
print("y shape ", y.shape)
print(tokenizer.decode(X.detach().numpy()[0]))
print(tokenizer.decode(y.detach().numpy()[0]))

NameError: name 'TokenDataset' is not defined

In [30]:
model3 = GPTModel3(n_vocab, embed_dim=embed_dim, context_size=seq_len)
print(summary(model3))

Layer (type:depth-idx)                   Param #
GPTModel3                                --
├─Embedding: 1-1                         3,216,448
├─Embedding: 1-2                         512
├─LayerNorm: 1-3                         128
├─Linear: 1-4                            4,096
├─Linear: 1-5                            4,096
├─Linear: 1-6                            4,096
├─Linear: 1-7                            4,160
├─Linear: 1-8                            3,216,448
Total params: 6,449,984
Trainable params: 6,449,984
Non-trainable params: 0


In [41]:
y, (mask, qk_softmaxed) = model3(X)
print(y.shape)
print(mask[0])
print(qk_softmaxed[0])


torch.Size([5, 8, 50257])
tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5760, 0.4240, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3356, 0.2582, 0.4062, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1622, 0.3186, 0.2987, 0.2205, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1553, 0.2938, 0.1981, 0.1989, 0.1539, 0.0000, 0.0000, 0.0000],
        [0.2511, 0.1020, 0.1513, 0.0942, 0.1886, 0.2128, 0.0000, 0.0000],
        [0.1574, 0.1318, 0.1266, 0.1219, 0.1247, 0.2224, 0.1152, 0.0000],
        [0.0842, 0.0920, 0.1234, 0.0920, 0.1103, 0.1265, 0.2482, 0.1235]],
       grad_fn=<SelectBackward0>)


In [69]:
# generate
inp = tokenizer.encode("Hello world ")
inp = torch.tensor(inp)
print("Shape of inp: ", inp.shape)
inp = inp.unsqueeze(0)
print("Shape of inp after unsqueeze: ", inp.shape)
op = model3.generate(inp)
print("Shape of op: ", op.shape)
tokenizer.decode(op[0].tolist())


Shape of inp:  torch.Size([3])
Shape of inp after unsqueeze:  torch.Size([1, 3])
Shape of op:  torch.Size([1, 33])


'Hello world KeeperSim Tong AKamate greedyheet Foundingachel messICANhall ubiquVL discriminatory tipping Scandvable manufactureryrus►rawdownloadcloneembedreportprintNINGolls transm George Galile Arcinitely�'